# Episode 27: Testing

Every code example so far has made a real API call — slow, costs money, and gives a *different* answer each run. That's fine for a workshop. It's a disaster for a test suite.

**In this episode you'll build:** deterministic tests for your agents with `TestConfig` — no API calls, no cost, instant and repeatable.

## Why agents are hard to test

A normal unit test asserts `f(2) == 4`. But an agent calls an LLM, and the LLM:

- is **slow** — every test waits on a network round-trip
- **costs money** — a CI run with hundreds of tests adds up
- is **non-deterministic** — the same input can produce different words

`TestConfig` solves all three. It's a config you pass *in place of* a model config — instead of calling an LLM, it replays a queue of responses you supply. Your test becomes a normal, fast, free, deterministic unit test.

## Mocking a response

Pass `TestConfig("...")` as the `config` and the agent returns exactly that string. No network, no key required.

In [ ]:
from autogen.beta import Agent
from autogen.beta.testing import TestConfig

agent = Agent("assistant")

reply = await agent.ask("Hi!", config=TestConfig("This is a mocked response."))

assert reply.body == "This is a mocked response."
print("PASS — mocked response returned, no API call made")

## Testing a tool-call flow

Real agent runs often involve a tool call: the LLM asks for a tool, the tool runs, the LLM answers. `TestConfig` replays that whole loop — pass a `ToolCallEvent` as the first response, then the final answer string.

The tool still executes for real; only the LLM's decisions are mocked.

In [ ]:
from autogen.beta.events import ToolCallEvent


def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"{city}: 21C, sunny."


weather_bot = Agent("weather_bot", tools=[get_weather])

test_config = TestConfig(
    ToolCallEvent(
        name="get_weather", arguments='{"city": "Paris"}'
    ),  # turn 1: call the tool
    "The weather in Paris is 21C and sunny.",  # turn 2: final answer
)

reply = await weather_bot.ask("What's the weather in Paris?", config=test_config)

assert reply.body == "The weather in Paris is 21C and sunny."
print("PASS — tool-call flow produced the expected final answer")

## Testing *your* code

The real win: `TestConfig` lets you test the functions *you* build on top of agents. Here `triage()` classifies a support message. We give each case its own mocked config and assert the function behaves — fast, free, and identical every run.

In [ ]:
async def triage(message: str, agent: Agent) -> str:
    """Classify a support message into a category."""
    reply = await agent.ask(f"Classify into BILLING or TECH: {message}")
    return reply.body.strip()


cases = [
    ("I was double charged", "BILLING"),
    ("How do I reset my password", "TECH"),
]

for message, expected in cases:
    bot = Agent(
        "triage_bot", prompt="Classify support messages.", config=TestConfig(expected)
    )
    result = await triage(message, bot)
    assert result == expected
    print(f"PASS — {message!r} -> {result}")

print("all triage tests passed")

## In a real test suite

In practice these go in `pytest` files. `TestConfig` needs no special fixture — it's just a config:

```python
# test_triage.py
import pytest

from autogen.beta import Agent
from autogen.beta.testing import TestConfig

from myapp import triage


@pytest.mark.asyncio
async def test_triage_billing():
    bot = Agent("triage_bot", prompt="Classify support messages.", config=TestConfig("BILLING"))
    assert await triage("I was double charged", bot) == "BILLING"
```

Run it with `pytest` — the whole file finishes in milliseconds and never touches the network. Add `pytest` and `pytest-asyncio` to your dev dependencies.

## Additional content

**Testing errors.** If a tool raises, the exception propagates out of `ask()` — assert it with `pytest.raises(...)`. If the LLM (mock) calls an unregistered tool, you get a `ToolNotFoundError`. Both let you test failure paths deterministically.

**One config per conversation turn.** Each argument to `TestConfig` is one LLM turn. A multi-step run needs as many responses as it has turns — a `ToolCallEvent` plus a final string is two turns.

**Mock the LLM, not the tools.** `TestConfig` only mocks the *model*. Your tools run for real — so a tool-flow test still verifies your tool code, just with the LLM's choices pinned.

## What's next

Tests keep agents correct. Episode 28 keeps them *affordable* — **cost optimization**: token limits, usage monitoring, and history compaction.